# AfriCoT-VLA: `africot_full` experiment grid on GPU

This notebook runs the full controller (slow loop with the Qwen2.5-VL-7B-Instruct VLAM on a GPU, latent world model, CEM planner and safety shield) on the four scenarios and the evaluation seeds 0 to 4.

* Results and explainability logs are written to Google Drive, so the run survives a disconnect.
* The grid is resumable: a run whose result file already exists is skipped. Re-run the grid cell after a disconnect.
* Every result file records the VLAM backend (`hf`) and the model id. Mock-backend runs are never reported.

Runtime: choose a GPU runtime (Runtime > Change runtime type). The 7B model in bfloat16 needs about 17 GB of GPU memory (A100 or L4).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/africot'
import os
os.makedirs(f'{DRIVE}/results', exist_ok=True)
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
os.makedirs(f'{DRIVE}/hf_cache', exist_ok=True)

Clone the repository at the shared build branch. If the repository is private, use a GitHub token in the URL or upload the repository to Drive and change `REPO`.

In [ ]:
REPO = '/content/africot-vla'
if not os.path.exists(REPO):
    !git clone -b claude/africot-build https://github.com/Topmost2020/africot-vla.git {REPO}
%cd {REPO}
!git log --oneline -1

Install the requirements, keeping Colab's own CUDA build of PyTorch (the repository pins the CPU wheels for local runs).

In [ ]:
!grep -v -E '^(torch|torchvision)==|extra-index-url' requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
import torch, sumo
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('SUMO_HOME', sumo.SUMO_HOME)

In [ ]:
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'          # model download cached on Drive
os.environ['AFRICOT_LOG_DIR'] = f'{DRIVE}/logs'      # explainability logs on Drive
!python -m pytest -q tests/test_smoke.py tests/test_subgoal.py

Run the grid. Resumable: re-run this cell after a disconnect and finished runs are skipped. `--jobs 1` because all runs share one GPU.

In [ ]:
!python run_grid.py --controllers africot_full --scenarios normal market incident outage \
    --seeds 0 1 2 3 4 --vlam-backend hf --vlam-model Qwen/Qwen2.5-VL-7B-Instruct \
    --results-dir {DRIVE}/results --jobs 1

Check what was produced: backend, model, slow-loop calls, parse failures and mean VLAM latency per run.

In [ ]:
import glob, json
for f in sorted(glob.glob(f'{DRIVE}/results/africot_full__*.json')):
    d = json.load(open(f)); m = d['metrics']
    print(os.path.basename(f), d['vlam_backend'], d['config'].get('vlam_model'), 'AWT %.1f' % m['awt'],
          'calls', m.get('slow_loop_calls'), 'parse failures', m.get('vlam_parse_failures'),
          'latency %.1f s' % (d['info'].get('vlam_latency_mean_s') or float('nan')))

Copy the result files into the repository's `results/` folder (download them, commit them on the build branch), then regenerate the tables and figures together with the baseline and ablation results:

In [ ]:
!cp {DRIVE}/results/africot_full__*.json results/
!python plots.py
!cat results/summary.md